# 03. Plate Detection Dataset (U6)

B00 영상의 다섯 동작 구간에서 프레임을 분산 추출하고, 들어 올리는 원판 하나를 수동 bbox로 표시해 YOLO 형식 학습 데이터를 만듭니다.

## 범위와 완료 기준

이 노트북은 **프레임 추출 → 수동 ROI 라벨링 → YOLO 변환 → contact sheet 검토**까지만 담당합니다. 아직 모델을 학습하거나 검증 성능을 주장하지 않습니다. 같은 이미지 풀을 `train`과 `val`에 연결한 YAML은 다음 단계의 파이프라인 smoke test 용도입니다.

## 1. Setup

재사용 로직은 `plate.py`, 영상 순회는 `video.py`에서 가져옵니다.

In [ ]:
from pathlib import Path
import os

import cv2
import matplotlib.pyplot as plt
import pandas as pd

from snatch_analysis.config import ARTIFACT_ROOT, B00_VIDEO_ID
from snatch_analysis.plate import (
    SAMPLE_PHASES,
    sample_phase_frames,
    save_plate_image,
    save_yolo_plate_label,
    yolo_bbox_to_opencv,
)
from snatch_analysis.video import iter_video_frames, read_video_properties


PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise FileNotFoundError(
        "프로젝트 루트 또는 notebooks 디렉터리에서 실행해 주세요."
    )

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

## 2. 조정할 설정

처음에는 `SAMPLE_COUNT=30`으로 유지하세요. 프레임을 확인한 뒤 라벨링할 때만 `RUN_MANUAL_LABELING=True`로 바꿉니다. 라벨 하나를 고칠 때는 `LABEL_FRAME_INDICES={프레임 번호}`와 `OVERWRITE_LABELS=True`를 함께 설정합니다. 모든 bbox를 contact sheet로 확인한 뒤에만 `CONTACT_SHEET_REVIEWED=True`로 바꿉니다.

In [ ]:
VIDEO_ID = B00_VIDEO_ID
SAMPLE_COUNT = 30  # 허용 범위: 20~40
RUN_MANUAL_LABELING = False
OVERWRITE_IMAGES = False
OVERWRITE_LABELS = False
LABEL_FRAME_INDICES = None  # 예: {65}; None이면 아직 라벨 없는 전체 샘플
CONTACT_SHEET_REVIEWED = False

MANIFEST_PATH = Path("data/metadata/videos.csv")
PHASE_SNAPSHOT_PATH = ARTIFACT_ROOT / VIDEO_ID / "phase_by_frame.csv"
DATASET_ROOT = Path("data/plate_dataset")
IMAGES_DIRECTORY = DATASET_ROOT / "images"
LABELS_DIRECTORY = DATASET_ROOT / "labels"
SAMPLE_MANIFEST_PATH = DATASET_ROOT / "sample_frames.csv"
DATASET_YAML_PATH = DATASET_ROOT / "dataset.yaml"
CONTACT_SHEET_PATH = ARTIFACT_ROOT / VIDEO_ID / "plate_label_contact_sheet.png"

if not 20 <= SAMPLE_COUNT <= 40:
    raise ValueError("SAMPLE_COUNT는 20~40이어야 합니다.")
if OVERWRITE_LABELS and not LABEL_FRAME_INDICES:
    raise ValueError(
        "덮어쓸 LABEL_FRAME_INDICES를 한 개 이상 지정하세요."
    )

## 3. 입력 검증

Manifest의 B00 경로와 U3에서 저장한 프레임별 구간표가 실제 영상과 일치하는지 확인합니다.

In [ ]:
manifest = pd.read_csv(MANIFEST_PATH)
video_rows = manifest.loc[manifest["video_id"] == VIDEO_ID]
if len(video_rows) != 1:
    raise ValueError(f"Manifest에서 {VIDEO_ID} 한 행을 찾을 수 없습니다.")

video_path = Path(video_rows.iloc[0]["path"])
video_properties = read_video_properties(video_path)
if video_properties["read_status"] != "ok":
    raise RuntimeError(f"영상을 읽을 수 없습니다: {video_path}")

phase_by_frame = pd.read_csv(PHASE_SNAPSHOT_PATH)
frame_count = int(video_properties["frame_count"])
if len(phase_by_frame) != frame_count:
    raise ValueError("구간표 행 수와 영상 frame_count가 다릅니다.")
if set(phase_by_frame["video_id"]) != {VIDEO_ID}:
    raise ValueError("구간표 video_id가 선택 영상과 다릅니다.")

pd.DataFrame(
    [{
        "video_id": VIDEO_ID,
        "video_path": video_path.as_posix(),
        **video_properties,
        "phase_rows": len(phase_by_frame),
    }]
)

## 4. 다섯 구간 분산 샘플링

각 구간 안에서 시간 순으로 고르게 선택하고 중복 프레임을 허용하지 않습니다. B00의 `bar_off_floor=0` 때문에 분석 구간상 Set-up 프레임이 비어 있으므로, **샘플링에서만 frame 0을 Set-up 기준 프레임으로 사용**합니다. 원래 `analysis_phase` 값은 바꾸지 않습니다.

In [ ]:
sampled_frames = sample_phase_frames(
    phase_by_frame,
    sample_count=SAMPLE_COUNT,
)
sampled_frames["image_filename"] = sampled_frames["frame_index"].map(
    lambda index: f"{VIDEO_ID}_frame_{index:04d}.jpg"
)
sampled_frames["label_filename"] = sampled_frames["frame_index"].map(
    lambda index: f"{VIDEO_ID}_frame_{index:04d}.txt"
)
if LABEL_FRAME_INDICES is not None:
    unknown_label_frames = set(LABEL_FRAME_INDICES) - set(
        sampled_frames["frame_index"]
    )
    if unknown_label_frames:
        raise ValueError(
            f"샘플에 없는 LABEL_FRAME_INDICES: {sorted(unknown_label_frames)}"
        )

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
sampled_frames.to_csv(SAMPLE_MANIFEST_PATH, index=False)

phase_distribution = (
    sampled_frames["sampling_phase"]
    .value_counts()
    .reindex(SAMPLE_PHASES, fill_value=0)
    .rename_axis("sampling_phase")
    .reset_index(name="sample_count")
)
display(phase_distribution)
sampled_frames

## 5. 프레임 추출

영상 전체를 메모리에 올리지 않고 한 번만 순회합니다. 기존 이미지는 기본적으로 유지합니다.

In [ ]:
sample_by_index = sampled_frames.set_index("frame_index")
pending_indices = {
    frame_index
    for frame_index, row in sample_by_index.iterrows()
    if OVERWRITE_IMAGES
    or not (IMAGES_DIRECTORY / row["image_filename"]).exists()
}
extraction_actions = [
    {"frame_index": frame_index, "action": "skipped"}
    for frame_index in sample_by_index.index
    if frame_index not in pending_indices
]
frame_iterator = (
    iter_video_frames(video_path, expected_frame_count=frame_count)
    if pending_indices
    else ()
)

for frame_index, _, frame in frame_iterator:
    if frame_index not in pending_indices:
        continue
    image_path = IMAGES_DIRECTORY / sample_by_index.loc[
        frame_index, "image_filename"
    ]
    action = save_plate_image(
        image_path,
        frame,
        overwrite=OVERWRITE_IMAGES,
    )
    extraction_actions.append(
        {"frame_index": frame_index, "action": action}
    )
    pending_indices.remove(frame_index)
    if not pending_indices:
        break

if pending_indices:
    raise RuntimeError(f"추출하지 못한 프레임: {sorted(pending_indices)}")
if len(extraction_actions) != SAMPLE_COUNT:
    raise RuntimeError("일부 샘플 프레임을 추출하지 못했습니다.")

pd.DataFrame(extraction_actions)["action"].value_counts().rename("count")

## 6. OpenCV 수동 bbox 라벨링

설정 셀에서 `RUN_MANUAL_LABELING=True`로 바꾼 뒤 이 셀을 실행하세요. 각 창에서 **선수가 들어 올리는, 카메라 쪽 원판 하나**만 드래그하고 `Enter` 또는 `Space`를 누릅니다. `c`는 현재 프레임을 건너뜁니다. 배경의 원판, 거울 속 원판, 반대쪽 원판은 라벨링하지 않습니다.

In [ ]:
label_actions = []

if RUN_MANUAL_LABELING:
    for row in sampled_frames.itertuples(index=False):
        if (
            LABEL_FRAME_INDICES is not None
            and row.frame_index not in LABEL_FRAME_INDICES
        ):
            continue
        image_path = IMAGES_DIRECTORY / row.image_filename
        label_path = LABELS_DIRECTORY / row.label_filename
        if label_path.exists() and not OVERWRITE_LABELS:
            label_actions.append(
                {"frame_index": row.frame_index, "action": "skipped"}
            )
            continue

        frame = cv2.imread(str(image_path))
        if frame is None:
            raise RuntimeError(f"추출 이미지를 읽을 수 없습니다: {image_path}")
        window_name = f"plate | frame {row.frame_index} | {row.sampling_phase}"
        bbox = cv2.selectROI(
            window_name,
            frame,
            showCrosshair=True,
            fromCenter=False,
        )
        cv2.destroyWindow(window_name)
        if bbox[2] == 0 or bbox[3] == 0:
            action = "cancelled"
        else:
            action = save_yolo_plate_label(
                label_path,
                bbox,
                image_width=frame.shape[1],
                image_height=frame.shape[0],
                overwrite=OVERWRITE_LABELS,
            )
        label_actions.append(
            {"frame_index": row.frame_index, "action": action}
        )
    cv2.destroyAllWindows()
else:
    print("수동 라벨링 대기: RUN_MANUAL_LABELING=True로 바꾸세요.")

pd.DataFrame(label_actions)

## 7. 모든 bbox contact sheet 검토

저장된 YOLO 좌표를 다시 pixel bbox로 복원해 한 화면에서 확인합니다. bbox가 원판을 벗어나거나 다른 물체를 가리키면 해당 프레임만 `OVERWRITE_LABELS=True`로 다시 라벨링하세요.

In [ ]:
contact_frames = []

for row in sampled_frames.itertuples(index=False):
    image_path = IMAGES_DIRECTORY / row.image_filename
    label_path = LABELS_DIRECTORY / row.label_filename
    if not label_path.exists():
        continue

    tokens = label_path.read_text(encoding="utf-8").strip().split()
    if len(tokens) != 5 or tokens[0] != "0":
        raise ValueError(f"잘못된 원판 라벨: {label_path}")
    yolo_bbox = tuple(float(value) for value in tokens[1:])

    frame = cv2.imread(str(image_path))
    if frame is None:
        raise RuntimeError(f"추출 이미지를 읽을 수 없습니다: {image_path}")
    x, y, width, height = yolo_bbox_to_opencv(
        yolo_bbox,
        image_width=frame.shape[1],
        image_height=frame.shape[0],
    )
    overlay = frame.copy()
    cv2.rectangle(overlay, (x, y), (x + width, y + height), (0, 255, 0), 4)
    thumbnail_width = 360
    thumbnail_height = round(overlay.shape[0] * thumbnail_width / overlay.shape[1])
    thumbnail = cv2.resize(overlay, (thumbnail_width, thumbnail_height))
    contact_frames.append((row, thumbnail))

if contact_frames:
    column_count = 5
    row_count = (len(contact_frames) + column_count - 1) // column_count
    figure, axes = plt.subplots(
        row_count,
        column_count,
        figsize=(20, 4 * row_count),
        squeeze=False,
    )
    for axis in axes.flat:
        axis.axis("off")
    for axis, (row, overlay) in zip(axes.flat, contact_frames):
        axis.imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
        axis.set_title(f"{row.frame_index} | {row.sampling_phase}")
    figure.tight_layout()
    CONTACT_SHEET_PATH.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(CONTACT_SHEET_PATH, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Contact sheet: {CONTACT_SHEET_PATH}")
else:
    print("저장된 라벨이 없어 contact sheet 생성을 기다립니다.")

## 8. YOLO dataset 설정과 수동 체크포인트

추가 YAML 라이브러리 없이 Ultralytics가 읽을 최소 설정만 저장합니다. U7로 넘어가기 전에 라벨 수가 샘플 수와 같고 contact sheet를 직접 확인해야 합니다.

In [ ]:
dataset_yaml = (
    f"path: {DATASET_ROOT.resolve().as_posix()}\n"
    "train: images\n"
    "val: images\n"
    "names:\n"
    "  0: plate\n"
)
DATASET_YAML_PATH.write_text(dataset_yaml, encoding="utf-8")

image_count = sum(
    (IMAGES_DIRECTORY / name).exists()
    for name in sampled_frames["image_filename"]
)
label_count = sum(
    (LABELS_DIRECTORY / name).exists()
    for name in sampled_frames["label_filename"]
)
if image_count != SAMPLE_COUNT or label_count != SAMPLE_COUNT:
    manual_checkpoint = "PENDING: 수동 bbox 라벨링 미완료"
elif not CONTACT_SHEET_PATH.exists() or not CONTACT_SHEET_REVIEWED:
    manual_checkpoint = "READY_FOR_REVIEW: contact sheet 직접 확인 필요"
else:
    manual_checkpoint = "PASS: 모든 bbox 직접 검토 완료"

pd.DataFrame(
    [{
        "sample_count": SAMPLE_COUNT,
        "image_count": image_count,
        "label_count": label_count,
        "manual_checkpoint": manual_checkpoint,
        "dataset_yaml": DATASET_YAML_PATH.as_posix(),
        "contact_sheet": (
            CONTACT_SHEET_PATH.as_posix()
            if CONTACT_SHEET_PATH.exists()
            else "N/A: 라벨링 대기"
        ),
    }]
)